# Cross-Validation Variance Analysis (Part 3/3) — Aggregator

**Authors:** Swagotam Malakar, Anamika Das, Dr. Ohidujjaman
**Environment:** Kaggle CPU (no GPU required), Internet not required
**Estimated runtime:** ~5 minutes (loads NB15a + 3 NB15b batch JSONs, plots, saves)

---

Part 3 of 3 — CPU-only aggregator combining NB15a (SMI + 4 classical) with NB15b1/b2/b3 (BanglaBERT) into a single 5-seed summary table and box plot. Uses NB1's single-seed F1 = 0.883 as a fallback value for any missing BanglaBERT batch.

See the **Kaggle Setup** cell below for required inputs and configuration.

## Kaggle Setup

Kaggle resets accelerator / internet / input settings after a notebook
re-import. Use this checklist before each run.

| Setting | Required value |
|---------|----------------|
| Accelerator | **None** (CPU only) |
| Internet | Off (not required) |
| Inputs | (1) NB15a output dataset containing `cv_variance_smi_classical_results.json`; (2) NB15b1 output dataset containing `cv_variance_banglabert_b1_results.json`; (3) NB15b2 output dataset containing `cv_variance_banglabert_b2_results.json`; (4) NB15b3 output dataset containing `cv_variance_banglabert_b3_results.json`. Items (2)-(4) are OPTIONAL — if any is missing, the aggregator falls back to the NB1 single-seed F1 = 0.883 for the missing seeds. |

**Add as Kaggle inputs:**
1. Right panel → **Add Input** → **Notebook Output** → search for your NB15a
   notebook → click **+**. Its `cv_variance_smi_classical_results.json`
   will appear under `/kaggle/input/<nb15a-slug>/`.
2. Repeat for NB15b1's notebook output. Its
   `cv_variance_banglabert_b1_results.json` will appear under
   `/kaggle/input/<nb15b1-slug>/`.
3. Repeat for NB15b2's notebook output. Its
   `cv_variance_banglabert_b2_results.json` will appear under
   `/kaggle/input/<nb15b2-slug>/`.
4. Repeat for NB15b3's notebook output. Its
   `cv_variance_banglabert_b3_results.json` will appear under
   `/kaggle/input/<nb15b3-slug>/`.
5. The notebook auto-discovers all four JSONs via `glob` over
   `/kaggle/input/**`, with local fallback paths for off-Kaggle
   development.

**Outputs (written to `/kaggle/working/`):**
- `cv_variance_final_summary.json` — canonical combined summary (all 6 models)
- `cv_variance_final_summary_table.csv` — flat results table
- `cv_variance_final_boxplot.png` — box plot of F1 across 5 seeds per model

No GPU, no internet, no HuggingFace model — pure CPU aggregation.


### 1. Environment Setup

In [1]:
# === 1. Environment Setup ===
import os, sys, time, json, warnings, glob
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # headless backend (Kaggle commit mode)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"pandas {pd.__version__}, numpy {np.__version__}", flush=True)
print(f"matplotlib {matplotlib.__version__}, seaborn {sns.__version__}", flush=True)
print("NB15c: Aggregator — combines NB15a (SMI + classical) + NB15b1/b2/b3 (BanglaBERT)", flush=True)


pandas 2.3.3, numpy 2.0.2
matplotlib 3.10.0, seaborn 0.13.2
NB15c: Aggregator — combines NB15a (SMI + classical) + NB15b1/b2/b3 (BanglaBERT)


### 2. Configuration

In [2]:
# === 2. Configuration ===

# Canonical filenames produced by NB15a and the three NB15b sub-notebooks
NB15A_JSON  = 'cv_variance_smi_classical_results.json'
NB15B1_JSON = 'cv_variance_banglabert_b1_results.json'   # seeds [42, 123]
NB15B2_JSON = 'cv_variance_banglabert_b2_results.json'   # seeds [2024, 7]
NB15B3_JSON = 'cv_variance_banglabert_b3_results.json'   # seed  [99]

# Mapping of batch -> seeds it should contain (for fallback reporting)
BATCH_SEEDS = {
    'b1': [42, 123],
    'b2': [2024, 7],
    'b3': [99],
}

# Single-seed reference F1 from NB1 (used if an NB15b batch is missing).
SINGLE_SEED_BANGLABERT_F1 = 0.8831

# Seeds (must match NB15a and the three NB15b sub-notebooks)
SEEDS = [42, 123, 2024, 7, 99]
N_FOLDS = 5

# Output paths
OUTPUT_JSON = OUTPUT_DIR / 'cv_variance_final_summary.json'
OUTPUT_CSV  = OUTPUT_DIR / 'cv_variance_final_summary_table.csv'
OUTPUT_PNG  = OUTPUT_DIR / 'cv_variance_final_boxplot.png'


def find_json(filename):
    """Search Kaggle input dirs + local paths for a JSON file."""
    # Kaggle input paths (recursive glob — handles nested notebook-output dirs)
    matches = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if matches:
        return matches[0]
    # Local paths (for development / testing)
    for local in [f'./{filename}', f'../results/{filename}', f'./results/{filename}',
                  f'/home/z/my-project/analysis/github_repo/results/{filename}',
                  f'/home/z/my-project/kaggle_outputs/{filename}',
                  f'/home/z/my-project/kaggle_outputs_v2/{filename}']:
        if os.path.isfile(local):
            return local
    return None


print(f"Searching for upstream JSONs...")
NB15A_PATH  = find_json(NB15A_JSON)
NB15B1_PATH = find_json(NB15B1_JSON)
NB15B2_PATH = find_json(NB15B2_JSON)
NB15B3_PATH = find_json(NB15B3_JSON)
print(f"  NB15a  output ({NB15A_JSON}):  {NB15A_PATH or 'NOT FOUND'}")
print(f"  NB15b1 output ({NB15B1_JSON}): {NB15B1_PATH or 'NOT FOUND (will use NB1 single-seed F1=0.883 for seeds 42, 123)'}")
print(f"  NB15b2 output ({NB15B2_JSON}): {NB15B2_PATH or 'NOT FOUND (will use NB1 single-seed F1=0.883 for seeds 2024, 7)'}")
print(f"  NB15b3 output ({NB15B3_JSON}): {NB15B3_PATH or 'NOT FOUND (will use NB1 single-seed F1=0.883 for seed 99)'}")

if NB15A_PATH is None:
    raise FileNotFoundError(
        f'\n========================================================\n'
        f'NB15a output not found: {NB15A_JSON}\n'
        f'\n'
        f'Add the NB15a notebook output as a Kaggle input:\n'
        f'  Right panel → "Add Input" → "Notebook Output" → select NB15a.\n'
        f'Or upload the JSON as a Kaggle dataset and add it as an input.\n'
        f'========================================================'
    )


Searching for upstream JSONs...
  NB15a  output (cv_variance_smi_classical_results.json):  /kaggle/input/notebooks/swagotammalakar/nb15a-cv-variance-smi-classical/cv_variance_smi_classical_results.json
  NB15b1 output (cv_variance_banglabert_b1_results.json): NOT FOUND (will use NB1 single-seed F1=0.883 for seeds 42, 123)
  NB15b2 output (cv_variance_banglabert_b2_results.json): NOT FOUND (will use NB1 single-seed F1=0.883 for seeds 2024, 7)
  NB15b3 output (cv_variance_banglabert_b3_results.json): NOT FOUND (will use NB1 single-seed F1=0.883 for seed 99)


### 3. Load NB15a Results (SMI + Classical)

In [3]:
# === 3. Load NB15a Results (SMI + Classical) ===
with open(NB15A_PATH, 'r') as f:
    nb15a = json.load(f)

print(f'Loaded NB15a output: {NB15A_PATH}')
print(f'  n_seeds: {nb15a["n_seeds"]}, seeds: {nb15a["seeds"]}, n_folds: {nb15a["n_folds_per_seed"]}')
print(f'  Models present: {list(nb15a["per_model_results"].keys())}')

# Extract per-model per-seed F1
nb15a_models = nb15a['per_model_results']
print(f'\nPer-model across-seed summary (from NB15a):')
for m, r in nb15a_models.items():
    print(f'  {m:<24} F1 = {r["mean"]:.4f} ± {r["std"]:.4f}  '
          f'(per-seed: {[round(f, 4) for f in r["per_seed_f1"]]})')


Loaded NB15a output: /kaggle/input/notebooks/swagotammalakar/nb15a-cv-variance-smi-classical/cv_variance_smi_classical_results.json
  n_seeds: 5, seeds: [42, 123, 2024, 7, 99], n_folds: 5
  Models present: ['SMI', 'Logistic Regression', 'Random Forest', 'Linear SVM', 'XGBoost']

Per-model across-seed summary (from NB15a):
  SMI                      F1 = 0.8081 ± 0.0044  (per-seed: [0.8091, 0.8071, 0.8008, 0.8144, 0.809])
  Logistic Regression      F1 = 0.7817 ± 0.0052  (per-seed: [0.7794, 0.7876, 0.7796, 0.7877, 0.7742])
  Random Forest            F1 = 0.7791 ± 0.0089  (per-seed: [0.7698, 0.792, 0.7848, 0.7687, 0.7802])
  Linear SVM               F1 = 0.7749 ± 0.0052  (per-seed: [0.7704, 0.7804, 0.7676, 0.7804, 0.7755])
  XGBoost                  F1 = 0.7386 ± 0.0123  (per-seed: [0.7549, 0.7473, 0.7222, 0.7417, 0.7271])


### 4. Load NB15b Batch Results (3 Batches, with NB1 Single-Seed Fallback)

In [4]:
# === 4. Load NB15b Batch Results (3 batches, with NB1 single-seed fallback) ===

# Each batch JSON uses a dict keyed by str(seed) for per_seed_f1 and per_seed_per_fold_f1.
# We merge all available batches into a single per-seed map.
banglabert_per_seed_f1 = {}           # str(seed) -> float
banglabert_per_seed_per_fold_f1 = {}  # str(seed) -> list of floats
banglabert_per_seed_detail = {}       # str(seed) -> {'per_fold_f1': [...], 'run_mean_f1': ..., ...}
banglabert_batch_sources = {}         # str(seed) -> 'NB15b1' / 'NB15b2' / 'NB15b3' / 'NB1 fallback'
batches_found = []
batches_missing = []

batch_files = [
    ('b1', NB15B1_PATH, NB15B1_JSON, 'NB15b1'),
    ('b2', NB15B2_PATH, NB15B2_JSON, 'NB15b2'),
    ('b3', NB15B3_PATH, NB15B3_JSON, 'NB15b3'),
]

for batch_id, batch_path, batch_filename, source_label in batch_files:
    if batch_path is None:
        print(f'  {source_label} output ({batch_filename}) NOT FOUND — '
              f'using NB1 single-seed F1={SINGLE_SEED_BANGLABERT_F1:.4f} for seeds {BATCH_SEEDS[batch_id]}')
        batches_missing.append(batch_id)
        for s in BATCH_SEEDS[batch_id]:
            banglabert_per_seed_f1[str(s)] = SINGLE_SEED_BANGLABERT_F1
            banglabert_per_seed_per_fold_f1[str(s)] = None  # no fold-level data
            banglabert_per_seed_detail[str(s)] = None
            banglabert_batch_sources[str(s)] = 'NB1 fallback'
        continue

    with open(batch_path, 'r') as f:
        batch = json.load(f)
    batches_found.append(batch_id)
    print(f'  Loaded {source_label} output: {batch_path}')
    print(f'    batch={batch.get("batch")}, seeds={batch.get("seeds")}, '
          f'completed_seeds={batch.get("completed_seeds")}')

    # Merge per-seed F1 from this batch (dict keyed by str(seed))
    batch_per_seed_f1 = batch.get('per_seed_f1', {}) or {}
    batch_per_seed_per_fold_f1 = batch.get('per_seed_per_fold_f1', {}) or {}
    batch_per_seed_detail = batch.get('per_seed_detail', {}) or {}

    for seed_str, f1_val in batch_per_seed_f1.items():
        if f1_val is None:
            # Seed was started but didn't finish — fall back to NB1
            banglabert_per_seed_f1[seed_str] = SINGLE_SEED_BANGLABERT_F1
            banglabert_per_seed_per_fold_f1[seed_str] = None
            banglabert_per_seed_detail[seed_str] = None
            banglabert_batch_sources[seed_str] = 'NB1 fallback (seed incomplete in batch)'
        else:
            banglabert_per_seed_f1[seed_str] = float(f1_val)
            banglabert_per_seed_per_fold_f1[seed_str] = batch_per_seed_per_fold_f1.get(seed_str)
            banglabert_per_seed_detail[seed_str] = batch_per_seed_detail.get(seed_str)
            banglabert_batch_sources[seed_str] = source_label

    # Any seed in this batch's plan that didn't appear in per_seed_f1? -> NB1 fallback
    for s in BATCH_SEEDS[batch_id]:
        if str(s) not in banglabert_per_seed_f1:
            banglabert_per_seed_f1[str(s)] = SINGLE_SEED_BANGLABERT_F1
            banglabert_per_seed_per_fold_f1[str(s)] = None
            banglabert_per_seed_detail[str(s)] = None
            banglabert_batch_sources[str(s)] = 'NB1 fallback (seed missing from batch)'

# Compute across-seed mean / std using whatever seeds we have (default = all 5)
banglabert_seeds_present = [s for s in SEEDS if str(s) in banglabert_per_seed_f1]
banglabert_f1_values = [banglabert_per_seed_f1[str(s)] for s in banglabert_seeds_present]

if banglabert_f1_values:
    banglabert_mean = float(np.mean(banglabert_f1_values))
    banglabert_std = float(np.std(banglabert_f1_values))
else:
    banglabert_mean = SINGLE_SEED_BANGLABERT_F1
    banglabert_std = 0.0

# Determine BanglaBERT source label
n_real_batches = len(batches_found)
n_fallback_seeds = sum(1 for v in banglabert_batch_sources.values()
                       if 'NB1 fallback' in v)
if n_real_batches == 0:
    banglabert_source = 'NB1 single-seed (all batches missing)'
    banglabert_n_seeds_completed = 1
elif n_fallback_seeds == 0:
    banglabert_source = 'NB15b (batches 1+2+3)'
    banglabert_n_seeds_completed = len(banglabert_seeds_present)
else:
    banglabert_source = f'NB15b (partial — {n_fallback_seeds} seed(s) NB1 fallback)'
    banglabert_n_seeds_completed = len(banglabert_seeds_present) - n_fallback_seeds

print(f'\nBanglaBERT merged summary:')
print(f'  Seeds present: {banglabert_seeds_present}')
print(f'  Per-seed F1: {[round(banglabert_per_seed_f1[str(s)], 4) for s in banglabert_seeds_present]}')
print(f'  Per-seed source: {banglabert_batch_sources}')
print(f'  Across-seed mean ± std: {banglabert_mean:.4f} ± {banglabert_std:.4f}')
print(f'  Source label: {banglabert_source}')
print(f'  Real (non-fallback) seeds: {banglabert_n_seeds_completed}')


  NB15b1 output (cv_variance_banglabert_b1_results.json) NOT FOUND — using NB1 single-seed F1=0.8795 for seeds [42, 123]
  NB15b2 output (cv_variance_banglabert_b2_results.json) NOT FOUND — using NB1 single-seed F1=0.8795 for seeds [2024, 7]
  NB15b3 output (cv_variance_banglabert_b3_results.json) NOT FOUND — using NB1 single-seed F1=0.8795 for seeds [99]

BanglaBERT merged summary:
  Seeds present: [42, 123, 2024, 7, 99]
  Per-seed F1: [0.8795, 0.8795, 0.8795, 0.8795, 0.8795]
  Per-seed source: {'42': 'NB1 fallback', '123': 'NB1 fallback', '2024': 'NB1 fallback', '7': 'NB1 fallback', '99': 'NB1 fallback'}
  Across-seed mean ± std: 0.8795 ± 0.0000
  Source label: NB1 single-seed (all batches missing)
  Real (non-fallback) seeds: 1


### 5. Combined Results Table

In [5]:
# === 5. Combined Results Table ===

rows = []

# --- BanglaBERT (from NB15b1/b2/b3 merged, with NB1 fallback) ---
rows.append({
    'Model': 'BanglaBERT',
    'Source': banglabert_source,
    **{f'Seed {s} F1': (round(banglabert_per_seed_f1[str(s)], 4)
                        if str(s) in banglabert_per_seed_f1 else 'n/a')
       for s in SEEDS},
    'Mean ± Std': f'{banglabert_mean:.4f} ± {banglabert_std:.4f}',
})

# --- SMI ---
smi = nb15a_models['SMI']
rows.append({
    'Model': 'SMI',
    'Source': 'NB15a',
    **{f'Seed {s} F1': round(v, 4) for s, v in zip(nb15a['seeds'], smi['per_seed_f1'])},
    'Mean ± Std': f'{smi["mean"]:.4f} ± {smi["std"]:.4f}',
})

# --- Classical models (preserve NB1's master_comparison.csv ordering) ---
classical_order = ['Random Forest', 'Logistic Regression', 'Linear SVM', 'XGBoost']
for model_name in classical_order:
    if model_name not in nb15a_models:
        continue
    r = nb15a_models[model_name]
    rows.append({
        'Model': model_name,
        'Source': 'NB15a',
        **{f'Seed {s} F1': round(v, 4) for s, v in zip(nb15a['seeds'], r['per_seed_f1'])},
        'Mean ± Std': f'{r["mean"]:.4f} ± {r["std"]:.4f}',
    })

results_df = pd.DataFrame(rows)
print('=' * 120)
print('CV VARIANCE FINAL SUMMARY — All Models (5×5-fold CV across 5 seeds)')
print('=' * 120)
print(results_df.to_string(index=False))

results_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved: {OUTPUT_CSV.name}')


CV VARIANCE FINAL SUMMARY — All Models (5×5-fold CV across 5 seeds)
              Model                                Source  Seed 42 F1  Seed 123 F1  Seed 2024 F1  Seed 7 F1  Seed 99 F1      Mean ± Std
         BanglaBERT NB1 single-seed (all batches missing)      0.8795       0.8795        0.8795     0.8795      0.8795 0.8795 ± 0.0000
                SMI                                 NB15a      0.8091       0.8071        0.8008     0.8144      0.8090 0.8081 ± 0.0044
      Random Forest                                 NB15a      0.7698       0.7920        0.7848     0.7687      0.7802 0.7791 ± 0.0089
Logistic Regression                                 NB15a      0.7794       0.7876        0.7796     0.7877      0.7742 0.7817 ± 0.0052
         Linear SVM                                 NB15a      0.7704       0.7804        0.7676     0.7804      0.7755 0.7749 ± 0.0052
            XGBoost                                 NB15a      0.7549       0.7473        0.7222     0.7417      0.7

### 6. Visualization — Box Plot of F1 Across 5 Seeds (All Models)

In [6]:
# === 6. Visualization — Box Plot of F1 Across 5 Seeds (All Models) ===

plot_rows = []
plot_model_order = []

# BanglaBERT — only plot per-seed points for seeds that came from a real
# NB15b batch (not the NB1 fallback). If no real seeds are available,
# show a single point at the NB1 reference value.
real_seeds = [s for s in SEEDS
              if str(s) in banglabert_batch_sources
              and 'NB1 fallback' not in banglabert_batch_sources[str(s)]]
if real_seeds:
    plot_model_order.append('BanglaBERT')
    for s in real_seeds:
        plot_rows.append({'Model': 'BanglaBERT', 'Seed': s,
                          'F1': banglabert_per_seed_f1[str(s)]})
else:
    plot_model_order.append('BanglaBERT (NB1 single-seed)')
    plot_rows.append({'Model': 'BanglaBERT (NB1 single-seed)', 'Seed': 42,
                      'F1': SINGLE_SEED_BANGLABERT_F1})

# SMI
plot_model_order.append('SMI')
for s, f1 in zip(nb15a['seeds'], smi['per_seed_f1']):
    plot_rows.append({'Model': 'SMI', 'Seed': s, 'F1': f1})

# Classical
for model_name in classical_order:
    if model_name not in nb15a_models:
        continue
    plot_model_order.append(model_name)
    r = nb15a_models[model_name]
    for s, f1 in zip(nb15a['seeds'], r['per_seed_f1']):
        plot_rows.append({'Model': model_name, 'Seed': s, 'F1': f1})

plot_df = pd.DataFrame(plot_rows)

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=plot_df, x='Model', y='F1', order=plot_model_order,
            ax=ax, color='#4ECDC4', width=0.5, showmeans=True,
            meanprops={'marker': 'D', 'markerfacecolor': 'red',
                       'markeredgecolor': 'red', 'markersize': 7})
sns.stripplot(data=plot_df, x='Model', y='F1', order=plot_model_order,
              ax=ax, color='black', size=4, jitter=True, alpha=0.7)
if real_seeds:
    title = (f'F1 Score Across 5 CV Seeds — All Models (5×5-fold CV)\n'
             f'(BanglaBERT from NB15b1/b2/b3, {banglabert_n_seeds_completed} '
             f'real seed(s), {len(SEEDS) - len(real_seeds)} NB1 fallback)')
else:
    title = ('F1 Score Across 5 CV Seeds — All Models (5×5-fold CV)\n'
             '(BanglaBERT from NB1 single-seed reference — run NB15b1/b2/b3 for full ablation)')
ax.set_title(title, fontsize=12)
ax.set_ylabel('F1 Score (per-seed mean of 5 folds)', fontsize=12)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylim(0.65, 0.92)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {OUTPUT_PNG.name}')


Saved: cv_variance_final_boxplot.png


### 7. Save Final Results

In [7]:
# === 7. Save Final Results (JSON) ===

# Build per-model results dict
per_model_results = {}

# BanglaBERT — merge per-seed detail from the three NB15b batches
banglabert_per_seed_f1_list = [banglabert_per_seed_f1[str(s)] for s in SEEDS
                               if str(s) in banglabert_per_seed_f1]
per_model_results['BanglaBERT'] = {
    'per_seed_f1': banglabert_per_seed_f1_list,
    'per_seed_f1_by_seed': banglabert_per_seed_f1,         # dict keyed by str(seed)
    'per_seed_per_fold_f1': banglabert_per_seed_per_fold_f1,
    'per_seed_detail': banglabert_per_seed_detail,
    'mean': banglabert_mean,
    'std': banglabert_std,
    'source': banglabert_source,
    'n_seeds_completed': banglabert_n_seeds_completed,
    'per_seed_source': banglabert_batch_sources,
    'batches_found':  batches_found,
    'batches_missing': batches_missing,
}

# SMI
per_model_results['SMI'] = {
    'per_seed_f1': smi['per_seed_f1'],
    'mean': smi['mean'],
    'std': smi['std'],
    'source': 'NB15a',
}

# Classical
for model_name in classical_order:
    if model_name not in nb15a_models:
        continue
    r = nb15a_models[model_name]
    per_model_results[model_name] = {
        'per_seed_f1': r['per_seed_f1'],
        'mean': r['mean'],
        'std': r['std'],
        'source': 'NB15a',
    }

# Key findings
key_findings = []

# 1. Single-seed representativeness check
single_seed_refs = {
    'BanglaBERT':            SINGLE_SEED_BANGLABERT_F1,
    'SMI':                   0.8091,
    'Random Forest':         0.7879,
    'Logistic Regression':   0.7771,
    'Linear SVM':            0.7636,
    'XGBoost':               0.7524,
}
for model_name, r in per_model_results.items():
    ref = single_seed_refs.get(model_name)
    if ref is None or r.get('std') is None or r.get('per_seed_f1') is None:
        key_findings.append(
            f"{model_name}: single-seed only ({r['source']}); across-seed variance not available."
        )
        continue
    is_representative = abs(ref - r['mean']) <= r['std']
    status = 'representative' if is_representative else 'an outlier'
    key_findings.append(
        f"{model_name}: across-seed F1 = {r['mean']:.4f} ± {r['std']:.4f} "
        f"(single-seed value {ref:.4f} is {status})."
    )

# 2. Highest-variance model
stds = {m: r['std'] for m, r in per_model_results.items() if r.get('std') is not None}
if stds:
    highest_var_model = max(stds, key=stds.get)
    key_findings.append(
        f"Highest across-seed variance: {highest_var_model} (std={stds[highest_var_model]:.4f})."
    )

# 3. Ranking stability (high → low by mean F1)
means = {m: r['mean'] for m, r in per_model_results.items() if r.get('mean') is not None}
if means:
    ranking = sorted(means.items(), key=lambda kv: -kv[1])
    key_findings.append(
        "Mean ranking across seeds (high → low): "
        + ", ".join(f"{m} ({v:.4f})" for m, v in ranking)
    )

# 4. BanglaBERT batch provenance summary
key_findings.append(
    f"BanglaBERT batches found: {batches_found}; missing: {batches_missing}. "
    f"Source label: '{banglabert_source}'."
)

results_json = {
    'n_seeds': len(SEEDS),
    'seeds': SEEDS,
    'n_folds_per_seed': N_FOLDS,
    'per_model_results': per_model_results,
    'banglabert_batch_summary': {
        'batches_found':   batches_found,
        'batches_missing': batches_missing,
        'per_seed_source': banglabert_batch_sources,
        'source_label':    banglabert_source,
        'n_real_seeds':    banglabert_n_seeds_completed,
        'batch_files': {
            'b1': NB15B1_JSON,
            'b2': NB15B2_JSON,
            'b3': NB15B3_JSON,
        },
        'batch_seeds': BATCH_SEEDS,
    },
    'single_seed_reference_f1': single_seed_refs,
    'key_findings': key_findings,
    'note': ('Part 3/3: Aggregator. Combines NB15a (SMI + classical) + '
             'NB15b1/b2/b3 (BanglaBERT, 3 batches). If any NB15b batch '
             'is missing, falls back to NB1 single-seed F1=0.883 for '
             'the seeds in that batch.'),
    'sources': {
        'NB15a_path':  NB15A_PATH,
        'NB15b1_path': NB15B1_PATH,
        'NB15b2_path': NB15B2_PATH,
        'NB15b3_path': NB15B3_PATH,
        'banglabert_source': banglabert_source,
    },
    'created_by': 'NB15c_CV_Variance_Aggregate.ipynb',
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(results_json, f, indent=2, default=str)

print(f'Saved: {OUTPUT_JSON.name}')
print(f'\nKey findings:')
for kf in key_findings:
    print(f'  • {kf}')


Saved: cv_variance_final_summary.json

Key findings:
  • BanglaBERT: across-seed F1 = 0.8795 ± 0.0000 (single-seed value 0.8795 is representative).
  • SMI: across-seed F1 = 0.8081 ± 0.0044 (single-seed value 0.8091 is representative).
  • Random Forest: across-seed F1 = 0.7791 ± 0.0089 (single-seed value 0.7879 is representative).
  • Logistic Regression: across-seed F1 = 0.7817 ± 0.0052 (single-seed value 0.7771 is representative).
  • Linear SVM: across-seed F1 = 0.7749 ± 0.0052 (single-seed value 0.7636 is an outlier).
  • XGBoost: across-seed F1 = 0.7386 ± 0.0123 (single-seed value 0.7524 is an outlier).
  • Highest across-seed variance: XGBoost (std=0.0123).
  • Mean ranking across seeds (high → low): BanglaBERT (0.8795), SMI (0.8081), Logistic Regression (0.7817), Random Forest (0.7791), Linear SVM (0.7749), XGBoost (0.7386)
  • BanglaBERT batches found: []; missing: ['b1', 'b2', 'b3']. Source label: 'NB1 single-seed (all batches missing)'.


### 8. Discussion

See the printed tables and the saved JSON (in the "Save Results" section) for full results. Key findings are recorded in the JSON's `key_findings` array.